# 01 — Ingest & Profile

Loads both sheets of `online_retail_II.xlsx` via `src/load.py`, produces a one-screen profile, and dumps a summary CSV that the write-up cites.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from load import load, summarise

df = load()
summary = summarise(df)
summary

LoadSummary(rows=1067371, sheets={'Year 2009-2010': 525461, 'Year 2010-2011': 541910}, date_min=Timestamp('2009-12-01 07:45:00'), date_max=Timestamp('2011-12-09 12:50:00'), unique_invoices=53628, unique_stock_codes=5304, unique_customers=5942, unique_countries=43)

In [2]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  string        
 1   StockCode    1067371 non-null  string        
 2   Description  1062989 non-null  string        
 3   Quantity     1067371 non-null  Int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   Int64         
 7   Country      1067371 non-null  string        
 8   sheet        1067371 non-null  object        
dtypes: Int64(2), datetime64[ns](1), float64(1), object(1), string(4)
memory usage: 350.1 MB


In [3]:
df.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Invoice,1067371,53628,537434,1350,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,1067371,5304,85123A,5829,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,1062989,5655,WHITE HANGING HEART T-LIGHT HOLDER,5918,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,1067371.0,<NA>,<NA>,<NA>,9.938898,-80995.0,1.0,3.0,10.0,80995.0,172.705794
InvoiceDate,1067371,NaN,NaN,NaN,2011-01-02 21:13:55.394028544,2009-12-01 07:45:00,2010-07-09 09:46:00,2010-12-07 15:28:00,2011-07-22 10:23:00,2011-12-09 12:50:00,NaN
Price,1067371.0,NaN,NaN,NaN,4.649388,-53594.36,1.25,2.1,4.15,38970.0,123.553059
Customer ID,824364.0,<NA>,<NA>,<NA>,15324.638504,12346.0,13975.0,15255.0,16797.0,18287.0,1697.46445
Country,1067371,43,United Kingdom,981330,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sheet,1067371,2,Year 2010-2011,541910,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Top 20 non-numeric stock codes: a quick eyeball at fees / adjustments / tests
mask = ~df['StockCode'].fillna('').str.match(r'^\d{5,6}[A-Za-z]?$')
df.loc[mask, 'StockCode'].value_counts().head(20)

StockCode
POST            2122
DOT             1446
M               1421
15056BL          923
C2               282
79323LP          232
D                177
79323GR          123
S                104
BANK CHARGES     102
15056bl           93
ADJUST            67
AMAZONFEE         43
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
CRUK              16
Name: count, dtype: Int64

In [5]:
df['Country'].value_counts().head(10)

Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Spain               3811
Switzerland         3189
Belgium             3123
Portugal            2620
Australia           1913
Name: count, dtype: Int64

In [6]:
import pandas as pd
rows = [
    ('total_rows', summary.rows),
    ('sheet_2009_2010', summary.sheets.get('Year 2009-2010', 0)),
    ('sheet_2010_2011', summary.sheets.get('Year 2010-2011', 0)),
    ('date_min', summary.date_min),
    ('date_max', summary.date_max),
    ('unique_invoices', summary.unique_invoices),
    ('unique_stock_codes', summary.unique_stock_codes),
    ('unique_customers', summary.unique_customers),
    ('unique_countries', summary.unique_countries),
]
pd.DataFrame(rows, columns=['metric', 'value']).to_csv('../output/profile_summary.csv', index=False)
pd.DataFrame(rows, columns=['metric', 'value'])

,metric,value
0,total_rows,1067371
1,sheet_2009_2010,525461
2,sheet_2010_2011,541910
3,date_min,2009-12-01 07:45:00
4,date_max,2011-12-09 12:50:00
5,unique_invoices,53628
6,unique_stock_codes,5304
7,unique_customers,5942
8,unique_countries,43
